# Keyframe export test

This notebook extracts keyframes from one video and saves them to public/keyframes.
Set VIDEO_SOURCE and USE_MVIDEO in the next cell.

In [13]:
from pathlib import Path
from urllib.parse import urlparse
import os
import re
import sys
import tempfile

import cv2
import requests

ROOT = Path.cwd().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.services.keyframe import extract_keyframes
from src.models.mvideo import MVideo

VIDEO_SOURCE = "https://res.cloudinary.com/dqfutdhge/video/upload/v1773890563/vl5odunv0cnfotk3tcpe.mp4"  # URL or local path
USE_MVIDEO = True  # True: use MVideo pipeline (URL only). False: use extract_keyframes.

OUT_DIR = ROOT / "public/keyframes"
CLEAR_OUTPUT = True

print(OUT_DIR)


def is_url(value):
    return value.startswith("http://") or value.startswith("https://")


def sanitize_name(value):
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value)
    return value.strip("_") or "video"


def download_to_temp(url):
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    fd, tmp_path = tempfile.mkstemp(suffix=".mp4")
    with os.fdopen(fd, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    return tmp_path

d:\private\multimedia database system\public\keyframes


In [14]:
tmp_path = None
OUT_DIR.mkdir(parents=True, exist_ok=True)
if CLEAR_OUTPUT:
    for old in OUT_DIR.glob("*.jpg"):
        old.unlink()

if USE_MVIDEO:
    if not is_url(VIDEO_SOURCE):
        raise ValueError("USE_MVIDEO=True requires a URL")
    name_hint = Path(urlparse(VIDEO_SOURCE).path).stem or "video"
    mv = MVideo(VIDEO_SOURCE)
    keyframes_iter = [
        (i, mf.timestamp_sec, mf.frame) for i, mf in enumerate(mv.frames)
    ]
else:
    if is_url(VIDEO_SOURCE):
        tmp_path = download_to_temp(VIDEO_SOURCE)
        video_path = tmp_path
        name_hint = Path(urlparse(VIDEO_SOURCE).path).stem or "video"
    else:
        video_path = VIDEO_SOURCE
        if not Path(video_path).exists():
            raise FileNotFoundError(f"Video not found: {video_path}")
        name_hint = Path(video_path).stem

    keyframes = extract_keyframes(video_path)
    keyframes_iter = [
        (i, kf.timestamp_sec, kf.frame_bgr) for i, kf in enumerate(keyframes)
    ]

prefix = sanitize_name(name_hint)
for i, ts, frame_bgr in keyframes_iter:
    out_path = OUT_DIR / f"{prefix}_kf_{i:03d}_t{ts:.2f}.jpg"
    cv2.imwrite(str(out_path), frame_bgr)

print(f"Saved {len(keyframes_iter)} frames to {OUT_DIR}")

Saved 1 frames to d:\private\multimedia database system\public\keyframes


In [ ]:
import matplotlib.pyplot as plt

files = sorted(OUT_DIR.glob("*.jpg"))
print(f"Found {len(files)} files")
if files:
    preview = files[:6]
    cols = len(preview)
    fig, axes = plt.subplots(1, cols, figsize=(3 * cols, 3))
    if cols == 1:
        axes = [axes]
    for ax, path in zip(axes, preview):
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(path.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()

In [1]:
import os
import sys
from pathlib import Path

import cv2, numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()

ROOT = Path.cwd().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.mframe import MFrame
from src.utils.cosin import cosine_similarity

# Cấu hình Histogram (HSV)
HIST_BINS = tuple(int(x) for x in os.getenv("HIST_BINS", "8,4,4").split(","))
HIST_RANGES = [int(x) for x in os.getenv("HIST_RANGES", "0,180,0,256,0,256").split(",")]

# Cấu hình HOG
HOG_BINS = int(os.getenv("HOG_BINS", 9))
HOG_CELL_SIZE = int(os.getenv("HOG_CELL_SIZE", 8))
HOG_BLOCK_SIZE = int(os.getenv("HOG_BLOCK_SIZE", 2))
HOG_RESIZE = (128, 128)

# Cấu hình Texture (LBP + GLCM)
TEXTURE_LBP_BINS = int(os.getenv("TEXTURE_LBP_BINS", 256))
TEXTURE_GLCM_LEVELS = int(os.getenv("TEXTURE_GLCM_LEVELS", 8))
TEXTURE_GLCM_DISTANCE = int(os.getenv("TEXTURE_GLCM_DISTANCE", 1))

img1_path = ROOT / "public/testframes/lion2.jpg"
img2_path = ROOT / "public/testframes/elephen1.jpg"

img1 = cv2.imread(str(img1_path))
img2 = cv2.imread(str(img2_path))

print(img1.shape, img2.shape)

# resize về cùng kích thước trước khi so sánh
h, w = img1.shape[:2]
img2_rs = cv2.resize(img2, (w, h))

print("mean abs diff:", np.mean(np.abs(img1.astype(float) - img2_rs.astype(float))))


frame1 = MFrame(img1, 224, 224, frame_idx=0, timestamp_sec=0)
frame1.compute_his(HIST_BINS, HIST_RANGES)
frame1.compute_hog(HOG_BINS, HOG_CELL_SIZE, HOG_BLOCK_SIZE)
frame1.compute_texture(TEXTURE_LBP_BINS, TEXTURE_GLCM_LEVELS, TEXTURE_GLCM_DISTANCE)
frame1.compute_vec(0.2, 0.3, 0.5)

frame2 = MFrame(img2_rs, 224, 224, frame_idx=1, timestamp_sec=1)
frame2.compute_his(HIST_BINS, HIST_RANGES)
frame2.compute_hog(HOG_BINS, HOG_CELL_SIZE, HOG_BLOCK_SIZE)
frame2.compute_texture(TEXTURE_LBP_BINS, TEXTURE_GLCM_LEVELS, TEXTURE_GLCM_DISTANCE)
frame2.compute_vec(0.2, 0.3, 0.5)

print("cosine similarity: ", cosine_similarity(frame1.vec, frame2.vec))

(564, 1119, 3) (569, 1116, 3)
mean abs diff: 75.1379640721093
cosine similarity:  0.8791337030524004
